# Module 7 — Solow and the Sources of Growth
## Capital, Technology, and the Steady State

**From Smith to Simulation: Computing the Ideas that Built Economics**  
The University of Edinburgh · School of Economics

---

## Part I — The History

### Post-War Prosperity and an Unanswered Question

In the decades after 1945, the world economy experienced the most sustained period of growth in human history. Western Europe rebuilt from rubble. The United States doubled its living standards in a single generation. Japan transformed from wartime devastation into the world's second-largest economy. West Germany's *Wirtschaftswunder* (economic miracle) saw real GDP per capita triple between 1950 and 1970.

But where did all this growth come from? The classical economists — Smith, Ricardo, Marx — had theories of value and distribution, but no formal model of how economies grow over time. Keynes explained short-run fluctuations, but said little about the long-run trajectory. The question was wide open: **what are the sources of economic growth?**

### Robert Solow and the 1956 Model

In 1956, **Robert Solow** (1924–2023), a young economist at MIT, published a short paper — *"A Contribution to the Theory of Economic Growth"* — that would become one of the most influential in all of economics. His model was elegant in its simplicity:

1. **Output** is produced from **capital** (machines, factories) and **labour** (workers) using a production function.
2. A fixed fraction of output is **saved and invested**, adding to the capital stock.
3. Capital **depreciates** over time — machines wear out.
4. The production function exhibits **diminishing returns** to capital: the first factory in a country adds enormously to output; the thousandth adds relatively little.

From these ingredients, Solow derived a startling conclusion: **capital accumulation alone cannot sustain long-run growth.** Because of diminishing returns, each new unit of capital adds less output. Eventually, new investment just replaces depreciation, and growth stops. The economy settles at a **steady state**.

> *"All theory depends on assumptions which are not quite true. That is what makes it theory. The art of successful theorizing is to make the inevitable simplifying assumptions in such a way that the final results are not very sensitive."*  
> — Robert Solow, *Quarterly Journal of Economics* (1956)

### The Solow Residual: Technology as the True Engine

If capital accumulation cannot explain sustained growth, what can? Solow's empirical companion paper (1957) provided the answer. He decomposed US growth between 1909 and 1949 and found that capital and labour together explained only about **12.5%** of output growth. The remaining **87.5%** was a residual — a measure of our ignorance, later called **total factor productivity (TFP)** or simply the **Solow residual**.

This residual captures everything that makes an economy more productive beyond simply adding more capital and labour: **technological progress**, better management, improved institutions, education, and innovation. Solow's message was clear: *technology, not saving, is the true engine of long-run growth.*

### The Growth Miracles

Solow's framework made sense of the post-war miracles:

- **Germany and Japan** started the post-war era with their capital stocks destroyed but their human capital and institutional knowledge largely intact. With low capital-to-labour ratios, the returns to investment were enormous — exactly as Solow's model predicts. They grew rapidly by converging toward the steady state.
- **The Asian Tigers** (South Korea, Taiwan, Hong Kong, Singapore) followed a similar pattern from the 1960s onward, achieving growth rates that dwarfed the Western experience.

The model also predicted that this rapid growth would slow as countries approached their steady states — which is precisely what happened.

### Edinburgh Connection

The Scottish Enlightenment's commitment to empirical inquiry — from David Hume's insistence that knowledge must be grounded in observation, to Adam Smith's systematic collection of evidence about the wealth of nations — laid the intellectual groundwork for the kind of quantitative growth economics Solow pioneered. Solow's method of confronting theory with data, decomposing growth into measurable components, and letting the residual speak for itself embodies the empiricist tradition that Edinburgh's thinkers championed two centuries earlier. Today, the University of Edinburgh's economics department continues this tradition through its research on productivity, innovation, and the sources of growth in the Scottish and global economies.

Solow received the Nobel Prize in Economics in 1987 for his contributions to the theory of economic growth.

---

## Part II — The Computation

### Setting Up

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import fsolve

### The Production Function

Solow used a **Cobb-Douglas production function**:

$$Y = A \cdot K^\alpha \cdot L^{1-\alpha}$$

where:
- $Y$ is total output (GDP)
- $A$ is **total factor productivity** (technology)
- $K$ is the capital stock (machines, buildings, infrastructure)
- $L$ is the labour force (number of workers)
- $\alpha$ is the capital share of output (typically around 1/3)

Since we care about living standards — output **per worker** — we divide by $L$. Defining $y = Y/L$ (output per worker) and $k = K/L$ (capital per worker):

$$y = A \cdot k^\alpha$$

This is the **intensive form** of the production function. Note the diminishing returns: when $\alpha < 1$, doubling capital per worker less than doubles output per worker.

In [ ]:
# Parameters
A = 1.0       # total factor productivity
alpha = 1/3   # capital share

# Production function in per-worker terms
def production(k, A=A, alpha=alpha):
    """Output per worker y = A * k^alpha."""
    return A * k**alpha

# Visualise diminishing returns
k_vals = np.linspace(0, 50, 300)
y_vals = production(k_vals)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(k_vals, y_vals, color='steelblue', linewidth=2.5)
ax.set_xlabel('Capital per worker (k)', fontsize=12)
ax.set_ylabel('Output per worker (y)', fontsize=12)
ax.set_title('Production Function: $y = A \cdot k^{\\alpha}$ (Diminishing Returns)', fontsize=13)
ax.grid(True, alpha=0.3)

# Annotate diminishing returns
for k_point in [5, 15, 30]:
    y_point = production(k_point)
    ax.plot(k_point, y_point, 'ko', markersize=6)
    ax.annotate(f'k={k_point}, y={y_point:.2f}',
                xy=(k_point, y_point), xytext=(k_point + 2, y_point + 0.3),
                fontsize=10, arrowprops=dict(arrowstyle='->', color='gray'))

plt.tight_layout()
plt.show()

print(f"Going from k=5 to k=15 (adding 10 units of capital per worker):")
print(f"  Output increases by {production(15) - production(5):.3f}")
print(f"Going from k=30 to k=40 (also adding 10 units):")
print(f"  Output increases by {production(40) - production(30):.3f}")
print(f"\nThis is diminishing returns: same additional capital, less additional output.")

### The Fundamental Solow Equation

The heart of the Solow model is a single differential equation describing how capital per worker evolves over time:

$$\frac{dk}{dt} = s \cdot y - \delta \cdot k = s \cdot A \cdot k^\alpha - \delta \cdot k$$

where:
- $s$ is the **saving rate** (fraction of output saved and invested)
- $\delta$ is the **depreciation rate** (fraction of capital that wears out each period)

Two forces pull in opposite directions:
- **Investment** $s \cdot y = s \cdot A \cdot k^\alpha$ adds to the capital stock (concave, due to diminishing returns)
- **Depreciation** $\delta \cdot k$ erodes the capital stock (linear in $k$)

When investment exceeds depreciation, $k$ grows. When depreciation exceeds investment, $k$ shrinks. When they are equal, $k$ is constant — the **steady state**.

In [ ]:
# Parameters
s = 0.3       # saving rate (30% of output is saved)
delta = 0.05  # depreciation rate (5% of capital depreciates each year)

# Plot investment vs depreciation
k_vals = np.linspace(0, 80, 300)
investment = s * production(k_vals)
depreciation = delta * k_vals

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(k_vals, investment, color='steelblue', linewidth=2.5, label=f'Investment: $s \cdot A \cdot k^\\alpha$ (s={s})')
ax.plot(k_vals, depreciation, color='coral', linewidth=2.5, label=f'Depreciation: $\delta \cdot k$ ($\delta$={delta})')
ax.fill_between(k_vals, investment, depreciation,
                where=investment > depreciation, alpha=0.15, color='steelblue',
                label='Capital growing (investment > depreciation)')
ax.fill_between(k_vals, investment, depreciation,
                where=investment < depreciation, alpha=0.15, color='coral',
                label='Capital shrinking (depreciation > investment)')

# Find and mark the steady state
k_star = (s * A / delta) ** (1 / (1 - alpha))
y_star = production(k_star)
ax.plot(k_star, s * y_star, 'ko', markersize=10, zorder=5)
ax.annotate(f'Steady state\nk* = {k_star:.2f}',
            xy=(k_star, s * y_star), xytext=(k_star + 8, s * y_star + 0.3),
            fontsize=11, arrowprops=dict(arrowstyle='->', color='black'),
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', edgecolor='gray'))

ax.set_xlabel('Capital per worker (k)', fontsize=12)
ax.set_ylabel('Investment / Depreciation per worker', fontsize=12)
ax.set_title('The Solow Diagram: Investment vs. Depreciation', fontsize=13)
ax.legend(fontsize=10, loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Steady state capital per worker:  k* = {k_star:.4f}")
print(f"Steady state output per worker:   y* = {y_star:.4f}")
print(f"Steady state consumption per worker: c* = (1-s)*y* = {(1-s)*y_star:.4f}")

### Simulating the Solow Model: Convergence to the Steady State

Let's simulate the economy starting from an initial capital stock $k_0$ and watch it converge to the steady state. We use simple Euler integration:

$$k_{t+1} = k_t + \Delta t \cdot \left( s \cdot A \cdot k_t^\alpha - \delta \cdot k_t \right)$$

In [ ]:
def simulate_solow(k0, s, delta, A, alpha, T=200, dt=1.0):
    """Simulate the Solow model from initial capital k0."""
    k = np.zeros(T)
    k[0] = k0
    for t in range(T - 1):
        y = A * k[t]**alpha
        dk = s * y - delta * k[t]
        k[t + 1] = k[t] + dt * dk
    y = A * k**alpha
    c = (1 - s) * y
    return k, y, c

# Simulate from different starting points
T = 200
k_star_analytical = (s * A / delta) ** (1 / (1 - alpha))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
starting_points = [2.0, 10.0, 80.0]
colors = ['steelblue', 'coral', 'seagreen']
labels = ['Poor country (k0=2)', 'Middle country (k0=10)', 'Over-capitalised (k0=80)']

for k0, color, label in zip(starting_points, colors, labels):
    k, y, c = simulate_solow(k0, s, delta, A, alpha, T=T)
    axes[0].plot(range(T), k, color=color, linewidth=2, label=label)
    axes[1].plot(range(T), y, color=color, linewidth=2, label=label)
    axes[2].plot(range(T), c, color=color, linewidth=2, label=label)

for ax, ylabel, title in zip(axes,
    ['Capital per worker (k)', 'Output per worker (y)', 'Consumption per worker (c)'],
    ['Capital Accumulation', 'Output per Worker', 'Consumption per Worker']):
    ax.axhline(y=k_star_analytical if 'Capital' in title else
               production(k_star_analytical) if 'Output' in title else
               (1 - s) * production(k_star_analytical),
               color='black', linestyle='--', alpha=0.5, label='Steady state')
    ax.set_xlabel('Time (years)', fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Regardless of the starting point, all paths converge to k* = {k_star_analytical:.2f}.")
print(f"This is the fundamental stability property of the Solow model.")

### Finding the Steady State Analytically and with `fsolve`

At the steady state, $dk/dt = 0$, so:

$$s \cdot A \cdot k^{*\alpha} = \delta \cdot k^*$$

Solving analytically:

$$k^* = \left( \frac{s \cdot A}{\delta} \right)^{\frac{1}{1-\alpha}}$$

We can also find this numerically using `fsolve`.

In [ ]:
# Analytical steady state
k_star_analytical = (s * A / delta) ** (1 / (1 - alpha))
y_star_analytical = A * k_star_analytical**alpha
c_star_analytical = (1 - s) * y_star_analytical

print("=== Analytical Steady State ===")
print(f"k* = (s*A / delta)^(1/(1-alpha)) = ({s}*{A} / {delta})^(1/(1-{alpha}))")
print(f"k* = {k_star_analytical:.6f}")
print(f"y* = A * k*^alpha = {y_star_analytical:.6f}")
print(f"c* = (1-s) * y* = {c_star_analytical:.6f}")

# Numerical steady state using fsolve
def solow_equation(k):
    """dk/dt = s*A*k^alpha - delta*k. At steady state this equals zero."""
    return s * A * k**alpha - delta * k

k_star_numerical = fsolve(solow_equation, x0=10.0)[0]
y_star_numerical = A * k_star_numerical**alpha
c_star_numerical = (1 - s) * y_star_numerical

print(f"\n=== Numerical Steady State (fsolve) ===")
print(f"k* = {k_star_numerical:.6f}")
print(f"y* = {y_star_numerical:.6f}")
print(f"c* = {c_star_numerical:.6f}")

print(f"\nMatch: {np.allclose(k_star_analytical, k_star_numerical)}")

### Comparative Statics: How Parameters Shape the Steady State

One of the model's strengths is clear predictions about how changes in parameters affect long-run outcomes. Let's explore how the steady state responds to changes in the saving rate $s$, the depreciation rate $\delta$, and technology $A$.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# 1. Vary saving rate
s_range = np.linspace(0.05, 0.60, 100)
k_star_s = (s_range * A / delta) ** (1 / (1 - alpha))
y_star_s = A * k_star_s**alpha
c_star_s = (1 - s_range) * y_star_s

axes[0].plot(s_range, k_star_s, color='steelblue', linewidth=2, label='k*')
axes[0].plot(s_range, y_star_s, color='coral', linewidth=2, label='y*')
axes[0].plot(s_range, c_star_s, color='seagreen', linewidth=2, label='c*')
axes[0].axvline(x=s, color='gray', linestyle='--', alpha=0.5, label=f'Baseline s={s}')
axes[0].set_xlabel('Saving rate (s)', fontsize=11)
axes[0].set_ylabel('Steady state values', fontsize=11)
axes[0].set_title('Effect of Saving Rate', fontsize=12)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# 2. Vary depreciation rate
delta_range = np.linspace(0.01, 0.15, 100)
k_star_d = (s * A / delta_range) ** (1 / (1 - alpha))
y_star_d = A * k_star_d**alpha
c_star_d = (1 - s) * y_star_d

axes[1].plot(delta_range, k_star_d, color='steelblue', linewidth=2, label='k*')
axes[1].plot(delta_range, y_star_d, color='coral', linewidth=2, label='y*')
axes[1].plot(delta_range, c_star_d, color='seagreen', linewidth=2, label='c*')
axes[1].axvline(x=delta, color='gray', linestyle='--', alpha=0.5, label=f'Baseline $\delta$={delta}')
axes[1].set_xlabel('Depreciation rate ($\delta$)', fontsize=11)
axes[1].set_ylabel('Steady state values', fontsize=11)
axes[1].set_title('Effect of Depreciation Rate', fontsize=12)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

# 3. Vary technology
A_range = np.linspace(0.5, 3.0, 100)
k_star_A = (s * A_range / delta) ** (1 / (1 - alpha))
y_star_A = A_range * k_star_A**alpha
c_star_A = (1 - s) * y_star_A

axes[2].plot(A_range, k_star_A, color='steelblue', linewidth=2, label='k*')
axes[2].plot(A_range, y_star_A, color='coral', linewidth=2, label='y*')
axes[2].plot(A_range, c_star_A, color='seagreen', linewidth=2, label='c*')
axes[2].axvline(x=A, color='gray', linestyle='--', alpha=0.5, label=f'Baseline A={A}')
axes[2].set_xlabel('Technology (A)', fontsize=11)
axes[2].set_ylabel('Steady state values', fontsize=11)
axes[2].set_title('Effect of Technology', fontsize=12)
axes[2].legend(fontsize=9)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Key insights from comparative statics:")
print("  - Higher saving rate s => higher k* and y*, but c* peaks and then declines.")
print("  - Higher depreciation delta => lower k*, y*, and c* (capital wears out faster).")
print("  - Higher technology A => higher k*, y*, and c* (the only source of sustained improvement).")

### The Golden Rule: Maximising Steady-State Consumption

Notice that steady-state consumption $c^*$ does not monotonically increase with the saving rate. Saving more raises output, but it also means consuming a smaller share. There is an optimal saving rate — called the **Golden Rule** — that maximises $c^*$.

At the steady state:

$$c^* = y^* - \delta k^* = A \cdot k^{*\alpha} - \delta \cdot k^*$$

To maximise, take the derivative with respect to $k^*$ and set it to zero:

$$\frac{dc^*}{dk^*} = \alpha A \cdot k^{*\alpha - 1} - \delta = 0$$

So the **Golden Rule capital** is:

$$k^*_{GR} = \left( \frac{\alpha A}{\delta} \right)^{\frac{1}{1-\alpha}}$$

and the **Golden Rule saving rate** is simply $s_{GR} = \alpha$ (the capital share of output).

In [ ]:
# Golden Rule
s_golden = alpha
k_golden = (s_golden * A / delta) ** (1 / (1 - alpha))
y_golden = A * k_golden**alpha
c_golden = (1 - s_golden) * y_golden

print(f"=== The Golden Rule ===")
print(f"Golden Rule saving rate:  s_GR = alpha = {s_golden:.4f}")
print(f"Golden Rule capital:      k*_GR = {k_golden:.4f}")
print(f"Golden Rule output:       y*_GR = {y_golden:.4f}")
print(f"Golden Rule consumption:  c*_GR = {c_golden:.4f}")

# Compare with our baseline (s=0.3)
print(f"\nBaseline (s = {s}):")
print(f"  k* = {k_star_analytical:.4f}, y* = {y_star_analytical:.4f}, c* = {c_star_analytical:.4f}")
print(f"\nThe baseline saves too much (s={s} > s_GR={s_golden:.4f}).")
print(f"Consumption would be HIGHER if the economy saved less!")

In [ ]:
# Graphical representation of the Golden Rule
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Left panel: c* as a function of s
s_range = np.linspace(0.01, 0.99, 500)
k_star_range = (s_range * A / delta) ** (1 / (1 - alpha))
y_star_range = A * k_star_range**alpha
c_star_range = (1 - s_range) * y_star_range

axes[0].plot(s_range, c_star_range, color='seagreen', linewidth=2.5)
axes[0].axvline(x=s_golden, color='goldenrod', linewidth=2, linestyle='--',
                label=f'Golden Rule: $s_{{GR}}$ = $\\alpha$ = {s_golden:.3f}')
axes[0].plot(s_golden, c_golden, 'o', color='goldenrod', markersize=12, zorder=5)
axes[0].axvline(x=s, color='gray', linewidth=1.5, linestyle=':',
                label=f'Baseline: s = {s}')
axes[0].set_xlabel('Saving rate (s)', fontsize=12)
axes[0].set_ylabel('Steady-state consumption per worker (c*)', fontsize=12)
axes[0].set_title('The Golden Rule Saving Rate', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Right panel: Solow diagram at the Golden Rule
k_plot = np.linspace(0, 50, 300)
axes[1].plot(k_plot, production(k_plot), color='steelblue', linewidth=2.5,
             label='Output: $y = Ak^{\\alpha}$')
axes[1].plot(k_plot, delta * k_plot, color='coral', linewidth=2.5,
             label='Depreciation: $\\delta k$')
axes[1].plot(k_plot, s_golden * production(k_plot), color='goldenrod', linewidth=2,
             linestyle='--', label=f'GR investment: $s_{{GR}} \cdot y$ (s={s_golden:.3f})')

# Mark the Golden Rule steady state
axes[1].plot(k_golden, delta * k_golden, 'o', color='goldenrod', markersize=10, zorder=5)

# Show that c* is the gap between y and delta*k
axes[1].annotate('', xy=(k_golden, production(k_golden)),
                 xytext=(k_golden, delta * k_golden),
                 arrowprops=dict(arrowstyle='<->', color='seagreen', linewidth=2))
axes[1].text(k_golden + 1.5, (production(k_golden) + delta * k_golden) / 2,
             f'Max c* = {c_golden:.2f}', fontsize=11, color='seagreen',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', edgecolor='gray'))

axes[1].set_xlabel('Capital per worker (k)', fontsize=12)
axes[1].set_ylabel('Output / Investment / Depreciation', fontsize=12)
axes[1].set_title('Golden Rule: Where the Gap is Largest', fontsize=13)
axes[1].legend(fontsize=9, loc='upper left')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("At the Golden Rule, the marginal product of capital equals the depreciation rate:")
print(f"  MPK = alpha * A * k_GR^(alpha-1) = {alpha * A * k_golden**(alpha-1):.4f}")
print(f"  delta = {delta}")
print(f"  These are equal: {np.isclose(alpha * A * k_golden**(alpha-1), delta)}")

### Convergence: Poor Countries Grow Faster

One of the Solow model's most powerful predictions is **conditional convergence**: countries farther from their steady state grow faster. A poor country with low $k_0$ has high marginal returns to capital, so investment is very productive. A rich country near $k^*$ has low marginal returns, so growth is slow.

This helps explain Germany and Japan's rapid post-war recovery: they had low $k$ but the same steady state (similar technology and institutions), so they grew very fast.

In [ ]:
# Compare growth rates at different levels of k
T = 150
k0_values = [2, 5, 10, 20, 40]
colors_conv = plt.cm.viridis(np.linspace(0.1, 0.9, len(k0_values)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for k0, color in zip(k0_values, colors_conv):
    k, y, c = simulate_solow(k0, s, delta, A, alpha, T=T)
    # Growth rate of y: (y[t+1] - y[t]) / y[t]
    growth_rate = np.diff(y) / y[:-1] * 100  # in percent
    axes[0].plot(range(T), y, color=color, linewidth=2, label=f'k0 = {k0}')
    axes[1].plot(range(T - 1), growth_rate, color=color, linewidth=2, label=f'k0 = {k0}')

axes[0].axhline(y=y_star_analytical, color='black', linestyle='--', alpha=0.5, label='y*')
axes[0].set_xlabel('Time (years)', fontsize=11)
axes[0].set_ylabel('Output per worker (y)', fontsize=11)
axes[0].set_title('Convergence: All Paths Approach the Same y*', fontsize=12)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

axes[1].axhline(y=0, color='black', linestyle='-', alpha=0.3)
axes[1].set_xlabel('Time (years)', fontsize=11)
axes[1].set_ylabel('Growth rate of y (%)', fontsize=11)
axes[1].set_title('Growth Rates: Poorer Countries Grow Faster', fontsize=12)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(-5, 25)

plt.tight_layout()
plt.show()

# Compute initial growth rates
print("Initial growth rates of output per worker:")
for k0 in k0_values:
    y0 = A * k0**alpha
    dk = s * y0 - delta * k0
    dy = alpha * A * k0**(alpha - 1) * dk
    growth = dy / y0 * 100
    print(f"  k0 = {k0:3d}:  growth rate = {growth:.2f}% per year")
print(f"\nThis is the convergence prediction: poor countries (low k0) grow faster.")
print(f"It explains the post-war miracles of Germany, Japan, and the Asian Tigers.")

---

## Part III — Exercises

### Exercise 1 — Two-Country Convergence

Consider two countries that share the same parameters ($A = 1$, $\alpha = 1/3$, $s = 0.25$, $\delta = 0.05$) but start at different capital levels — inspired by post-war Germany and the United States.

**(a)** Compute the common steady state $k^*$, $y^*$, $c^*$ for these parameters.

**(b)** Simulate both countries for 150 years starting from $k_0^{\text{poor}} = 2$ (war-ravaged Germany) and $k_0^{\text{rich}} = 20$ (the United States). Plot capital per worker and output per worker over time on the same graph.

**(c)** Compute and plot the annual growth rate of output per worker for both countries. At what year does the growth rate of the poor country first fall below 1%?

**(d)** Define the "catch-up ratio" as $y_{\text{poor}} / y_{\text{rich}}$. Plot it over time. In approximately what year does the poor country reach 90% of the rich country's output per worker?

In [ ]:
# Your answer here


### Exercise 2 — The Solow Residual

Solow's growth accounting decomposes output growth into contributions from capital, labour, and technology (the residual).

Starting from $Y = A K^\alpha L^{1-\alpha}$, taking logs and differentiating:

$$\frac{\Delta Y}{Y} = \frac{\Delta A}{A} + \alpha \frac{\Delta K}{K} + (1-\alpha) \frac{\Delta L}{L}$$

So the **Solow residual** (TFP growth) is:

$$\frac{\Delta A}{A} = \frac{\Delta Y}{Y} - \alpha \frac{\Delta K}{K} - (1-\alpha) \frac{\Delta L}{L}$$

Here is stylised data for a country over five decades (average annual growth rates):

| Decade | $\Delta Y / Y$ | $\Delta K / K$ | $\Delta L / L$ |
|--------|----------------|----------------|----------------|
| 1950s  | 7.5%           | 9.0%           | 2.0%           |
| 1960s  | 6.0%           | 8.0%           | 1.5%           |
| 1970s  | 4.0%           | 5.5%           | 1.0%           |
| 1980s  | 3.0%           | 4.0%           | 1.0%           |
| 1990s  | 2.5%           | 3.0%           | 0.5%           |

Use $\alpha = 1/3$.

**(a)** For each decade, compute the contributions of capital growth, labour growth, and TFP growth to output growth.

**(b)** Create a stacked bar chart showing the three contributions for each decade.

**(c)** In which decade was TFP growth highest? What fraction of total growth did TFP account for in each decade?

**(d)** This pattern (high early growth driven by capital accumulation, slowing over time as TFP becomes more important) is characteristic of which real-world countries? Explain in 2-3 sentences how Solow's framework interprets this.

In [ ]:
# Your answer here


### Exercise 3 — Population Growth Extension

Solow's full model includes population (labour force) growth at rate $n$. When the labour force grows, capital must be spread among more workers. The fundamental equation becomes:

$$\frac{dk}{dt} = s \cdot A \cdot k^\alpha - (\delta + n) \cdot k$$

Now capital per worker is eroded by **both** depreciation and population growth ("capital dilution").

**(a)** Derive the new steady state analytically:

$$k^* = \left( \frac{s \cdot A}{\delta + n} \right)^{\frac{1}{1-\alpha}}$$

Compute $k^*$, $y^*$, $c^*$ for $A = 1$, $\alpha = 1/3$, $s = 0.3$, $\delta = 0.05$, and $n = 0.02$.

**(b)** Use `fsolve` to verify your analytical answer. Then plot the Solow diagram with the $(\delta + n) \cdot k$ line replacing the $\delta \cdot k$ line. Show both on the same graph for comparison.

**(c)** Simulate the model for $n = 0$, $n = 0.02$, and $n = 0.04$ (all other parameters equal). Plot the paths of $y$ over time. What does higher population growth do to steady-state living standards?

**(d)** Derive the new Golden Rule saving rate when $n > 0$. (Hint: it is still $s_{GR} = \alpha$.) Explain intuitively why population growth does not affect the Golden Rule saving rate.

In [ ]:
# Your answer here


---

## Part IV — Quiz

### Conceptual Questions

**Q1.** In Solow's model, why does capital accumulation alone fail to sustain long-run growth?

(a) Because governments prevent excessive investment  
(b) Because the production function has diminishing returns to capital  
(c) Because workers refuse to save more  
(d) Because depreciation rates always increase over time  

**Q2.** The "Solow residual" refers to:

(a) The output left over after paying workers and capital owners  
(b) The fraction of output growth not explained by growth in capital and labour  
(c) The depreciation of the capital stock  
(d) The gap between actual and potential GDP  

**Q3.** The post-war growth miracles of Germany and Japan are best explained by Solow's model as:

(a) Rapid convergence from far below steady state, where returns to capital are high  
(b) Permanently higher technology levels than other countries  
(c) Extremely high population growth  
(d) Abandonment of market economies  

**Q4.** In the Solow model, the steady state is the point where:

(a) Output per worker is maximised  
(b) The saving rate equals the depreciation rate  
(c) New investment exactly replaces depreciated capital (per worker)  
(d) Consumption per worker is zero  

**Q5.** Solow's key empirical finding (1957) was that approximately what share of US output growth was explained by TFP (the residual)?

(a) About 12%  
(b) About 40%  
(c) About 65%  
(d) About 87%  

### Computational Questions

**Q6.** In the Solow model with $y = Ak^\alpha$, the steady-state capital per worker is:

(a) $k^* = s \cdot A / \delta$  
(b) $k^* = (s \cdot A / \delta)^{1/(1-\alpha)}$  
(c) $k^* = (s / \delta)^{\alpha}$  
(d) $k^* = A \cdot (s / \delta)^\alpha$  

**Q7.** The Golden Rule saving rate that maximises steady-state consumption is:

(a) $s_{GR} = 1 - \alpha$  
(b) $s_{GR} = \delta$  
(c) $s_{GR} = \alpha$ (the capital share)  
(d) $s_{GR} = 0.5$ always  

**Q8.** If the saving rate $s$ increases permanently, the Solow model predicts:

(a) A permanently higher growth rate of output  
(b) A temporarily higher growth rate and a higher steady-state level of output  
(c) No change in output  
(d) A lower steady-state level of output  

**Q9.** With parameters $A = 1$, $\alpha = 1/3$, $s = 0.3$, $\delta = 0.05$, the steady-state $k^*$ is:

(a) $(0.3 / 0.05)^{3/2} = 6^{1.5} \approx 14.7$  
(b) $(0.3 / 0.05)^{1/3} \approx 1.82$  
(c) $0.3 / 0.05 = 6$  
(d) $(0.3 \times 0.05)^{1.5} \approx 0.0058$  

**Q10.** Adding population growth $n$ to the Solow model changes the steady state formula to:

(a) $k^* = (s \cdot A / \delta)^{1/(1-\alpha)}$ (unchanged)  
(b) $k^* = (s \cdot A / (\delta + n))^{1/(1-\alpha)}$  
(c) $k^* = (s \cdot A / (\delta \cdot n))^{1/(1-\alpha)}$  
(d) $k^* = ((s + n) \cdot A / \delta)^{1/(1-\alpha)}$  

### Quiz Answers

<details>
<summary>Click to reveal answers</summary>

**Q1.** (b) The production function $y = Ak^\alpha$ with $\alpha < 1$ has diminishing returns to capital. Each additional unit of capital adds less output, so eventually new investment can only replace what depreciates rather than adding to the capital stock.

**Q2.** (b) The Solow residual is the portion of output growth that cannot be accounted for by the growth of capital and labour inputs. It is often interpreted as a measure of technological progress or total factor productivity (TFP) growth.

**Q3.** (a) With their capital stocks destroyed by the war, Germany and Japan were far below their steady states. The Solow model predicts that countries far from steady state grow rapidly because the marginal product of capital is high. As they accumulated capital, diminishing returns gradually slowed growth — exactly as observed.

**Q4.** (c) The steady state occurs where $s \cdot A \cdot k^{*\alpha} = \delta \cdot k^*$, meaning gross investment exactly offsets capital depreciation. Capital per worker and output per worker are constant at this point.

**Q5.** (d) In his 1957 paper, Solow found that approximately 87.5% of the growth in US output per worker between 1909 and 1949 was attributable to the residual (technological progress), with only about 12.5% explained by capital deepening.

**Q6.** (b) Setting $s \cdot A \cdot k^{*\alpha} = \delta \cdot k^*$ and solving: $k^{*1-\alpha} = s A / \delta$, so $k^* = (sA/\delta)^{1/(1-\alpha)}$.

**Q7.** (c) The Golden Rule maximises $c^* = Ak^{*\alpha} - \delta k^*$. The first-order condition gives $\alpha A k^{*(\alpha-1)} = \delta$, which implies $s_{GR} = \alpha$. The optimal saving rate equals capital's share of output.

**Q8.** (b) A higher saving rate shifts the investment curve up, creating a new, higher steady state. During the transition, growth is faster than normal, but once the new steady state is reached, the growth rate returns to zero (in the basic model without technological progress). The level of output is permanently higher, but the growth rate is only temporarily higher.

**Q9.** (a) $k^* = (sA/\delta)^{1/(1-\alpha)} = (0.3/0.05)^{1/(1-1/3)} = 6^{3/2} = 6\sqrt{6} \approx 14.70$.

**Q10.** (b) With population growth at rate $n$, capital per worker is diluted by both depreciation and the growing workforce. The steady state condition becomes $sAk^{*\alpha} = (\delta + n)k^*$, giving $k^* = (sA/(\delta + n))^{1/(1-\alpha)}$.

</details>

---

## Further Reading

- Solow, R.M. "A Contribution to the Theory of Economic Growth," *Quarterly Journal of Economics* (1956).
- Solow, R.M. "Technical Change and the Aggregate Production Function," *Review of Economics and Statistics* (1957).
- Jones, C.I. *Introduction to Economic Growth*, Chapters 2-3 (an accessible textbook treatment).
- Romer, D. *Advanced Macroeconomics*, Chapter 1 (for a more mathematical treatment).
- Warsh, D. *Knowledge and the Wealth of Nations* (a narrative history of growth theory).

---

*Next week: Module 8 — beyond the Solow model. If technology drives growth, where does technology come from? Endogenous growth theory and the economics of innovation.*